# Task #1: Data Loading & Normalization

In [1]:
import requests
import hashlib
import pandas as pd

# Accessing Data

To remain compatible with Google Colab and personal development IDE's, data can be loaded by using requests. As data gets manipulated and saved over the course of this project, it is even more crucial to have one source of truth for every dataset.

In [21]:
def load_json_from_github(path_from_root: str, branch: str="main"):
    """
    Retrieves JSON files from the Accenture 1O repository
    :param path_from_root: the path to the file from the branch's root
    :param branch: the branch the file is located on, assumes "main" branch
    :return: the file as a json object
    """
    url = f"https://raw.githubusercontent.com/Break-Through-Tech/Accenture-1O-contract-review-challenge/{branch}/{path_from_root}"

    response = requests.get(url)
    response.raise_for_status()
    return response.json()

data = load_json_from_github(
    path_from_root="data/cuad/train_separate_questions.json",
    branch="setup-and-explore"
)
raw_data = data["data"]


# Normalizing Data

This is the first step for data preprocessing. In order to run NLP and ML techniques on the CUAD dataset, it must be normalized into a standardized structure. This structure is as follows:
```python
{
    "contracts": pd.DataFrame,
    "documents": pd.DataFrame,
    "categories": pd.DataFrame,
    "annotation_sets": pd.DataFrame,
    "spans": pd.DataFrame,
}
```
Contracts include...
- `contract_id`: an enumerated key
- `title`: the title of the contract

Documents include...
- `document_id`: an enumerated key
- `contract_id`: foreign key linked to `contracts.contract_id`
- `context`: the complete, unchanged contract text
- `context_group_id`: a hash code for the context

Categories include...
- `category_id`: the category name in snake case
- `category_name`: a human-readable formatted `category_id`
- `question`: the question asked to identify key clauses

Annotation Sets include...
- `annotation_set_id`: a key consisting of the `contract_id` and `category_id`
- `contract_id`: foreign key linked to `contracts.contract_id`
- `category_id`: foreign key linked to `categories.category_id`
- `is_impossible`: boolean indicating whether an answer exists to the asked question

Spans include...
- `span_id`: a key consisting of the `contract_id` and `category_id` and span number
- `annotation_set_id`: a foreign key linked to `annotation_sets.annotation_set_id`
- `source_qa_id`: the contract's original `title` and `category_id`
- `answer_text`: the identified clause in plain text
- `answer_start`: the starting index of the `answer_text`
- `answer_end`: the ending index of the `answer_text`

In [3]:
def normalize_cuad(data: dict) -> dict:
    """
    Normalizes the given CUAD dataset from nested JSON objects into separate actionable DataFrames
    :param data: list of document objects from the CUAD dataset
    :return: a single dictionary that contains separate DataFrames for contracts, documents, categories, annotation sets, and spans. These DataFrames are connected via 'foreign keys' (aka their IDs)
    """

    # assign contract ids for each contract in the dataset
    for i, item in enumerate(data):
        item["contract_id"] = f"contract_{i + 1:04d}"

    # get each document and their respective data (title, context, qas)
    documents = pd.json_normalize(
        data,
        record_path="paragraphs",
        meta=["title", "contract_id"]
    )[["contract_id", "title", "context"]]

    # assign ids and hashes
    documents["document_id"] = [f"document_{i + 1:04d}" for i in range(len(documents))]
    documents["context_group_id"] = documents["context"].apply(
        lambda c: "context_" + hashlib.md5(c.encode("utf-8")).hexdigest()[:10]
    )

    # create the contracts "table" and remove duplicates
    contracts = (
        documents[["contract_id", "title"]]
        .drop_duplicates(subset="contract_id")
        .reset_index(drop=True)
    )

    # ensures all indexes are correct
    documents = documents[
        ["document_id", "contract_id", "context", "context_group_id"]
    ].reset_index(drop=True)

    # get all Q&A objects
    qas = pd.json_normalize(
        data,
        record_path=["paragraphs", "qas"],
        meta=["title", "contract_id"]
    )

    qas["category_name"] = qas["question"].str.extract(f'"([^"]+)')
    qas["category_id"] = (
        qas["category_name"]
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    qas["annotation_set_id"] = qas["contract_id"] + "__" + qas['category_id'].str.lower()

    # get categories. each question typically corresponds to a category
    categories = (
        qas[["category_id", "category_name", "question"]]
        .drop_duplicates(subset="category_id")
        .reset_index(drop=True)
    )

    # get the annotation set: one contract paired with one clause category/question
    annotation_sets = (
        qas.groupby(
            ["annotation_set_id", "contract_id", "category_id"]
        , as_index=False)["is_impossible"]
        .all()
    )

    # get each span: one individual answer inside the Q&A's answer list
    spans = qas[["answers", "id", "annotation_set_id"]].explode("answers")
    spans = spans[spans["answers"].notna()].copy() # drops impossible rows

    # expand the answer fields to be included in the top-level columns
    answer_fields = pd.json_normalize(spans["answers"])
    spans = pd.concat([spans.drop(columns="answers"), answer_fields], axis=1)

    spans = spans.rename(columns={"id": "source_qa_id", "text": "answer_text"})
    spans["answer_end"] = spans["answer_start"] + spans["answer_text"].str.len()

    # create the span id
    num_spans = spans.groupby("annotation_set_id").cumcount()
    spans["span_id"] = (
        spans["annotation_set_id"] + "__span_" + num_spans.astype(str).str.zfill(3)
    )
    spans = spans[[
        "span_id",
        "annotation_set_id",
        "source_qa_id",
        "answer_text",
        "answer_start",
        "answer_end",
    ]].reset_index(drop=True)

    return {
        "contracts": contracts,
        "documents": documents,
        "categories": categories,
        "annotation_sets": annotation_sets,
        "spans": spans,
    }



# Demo/Visualization

Recommended to use PyCharm or an equivalent IDE plugin to see DataFrame statistics.

In [4]:
for i, item in enumerate(raw_data):
    item["contract_id"] = f"contract_{i + 1:04d}"

In [5]:
df_documents = pd.json_normalize(
    raw_data,
    record_path="paragraphs",
    meta=["title", "contract_id"]
)[["contract_id", "title", "context"]]
df_documents

,contract_id,title,context
0,contract_0001,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,contract_0002,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,contract_0003,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3,contract_0004,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4,contract_0005,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...
...,...,...,...
403,contract_0404,CcRealEstateIncomeFundadv_20181205_POS 8C_EX-9...,Exhibit 99(h)(3) WHOLESALE MARKETING AGREEMENT...
404,contract_0405,"BLUEROCKRESIDENTIALGROWTHREIT,INC_06_01_2016-E...","Exhibit 1.1 400,000 Shares BLUEROCK RESIDE..."
405,contract_0406,"TALLGRASSENERGY,LP_02_20_2020-EX-99.26-JOINT F...",Exhibit 26\n\nJOINT FILING AGREEMENT\n\nPursua...
406,contract_0407,KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1-PROM...,Exhibit 10.1\n\n\n\nPROMOTION AGREEMENT\n\nby ...


In [6]:
df_documents["document_id"] = [f"document_{i + 1:04d}" for i in range(len(df_documents))]
df_documents["context_group_id"] = df_documents["context"].apply(
    lambda c: "context_" + hashlib.md5(c.encode("utf-8")).hexdigest()[:10]
)
df_documents

,contract_id,title,context,document_id,context_group_id
0,contract_0001,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...,document_0001,context_41f7921a65
1,contract_0002,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...,document_0002,context_d545018697
2,contract_0003,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...,document_0003,context_8cc65b1516
3,contract_0004,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...,document_0004,context_3b2346a2bc
4,contract_0005,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...,document_0005,context_1d348d1f41
...,...,...,...,...,...
403,contract_0404,CcRealEstateIncomeFundadv_20181205_POS 8C_EX-9...,Exhibit 99(h)(3) WHOLESALE MARKETING AGREEMENT...,document_0404,context_089b029b55
404,contract_0405,"BLUEROCKRESIDENTIALGROWTHREIT,INC_06_01_2016-E...","Exhibit 1.1 400,000 Shares BLUEROCK RESIDE...",document_0405,context_d621f3c0b1
405,contract_0406,"TALLGRASSENERGY,LP_02_20_2020-EX-99.26-JOINT F...",Exhibit 26\n\nJOINT FILING AGREEMENT\n\nPursua...,document_0406,context_d46fd85099
406,contract_0407,KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1-PROM...,Exhibit 10.1\n\n\n\nPROMOTION AGREEMENT\n\nby ...,document_0407,context_dc81522bbe


In [7]:
df_contracts = (
    df_documents[["contract_id", "title"]]
    .drop_duplicates(subset="contract_id")
    .reset_index(drop=True)
)
df_contracts

,contract_id,title
0,contract_0001,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...
1,contract_0002,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A..."
2,contract_0003,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT
3,contract_0004,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...
4,contract_0005,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU..."
...,...,...
403,contract_0404,CcRealEstateIncomeFundadv_20181205_POS 8C_EX-9...
404,contract_0405,"BLUEROCKRESIDENTIALGROWTHREIT,INC_06_01_2016-E..."
405,contract_0406,"TALLGRASSENERGY,LP_02_20_2020-EX-99.26-JOINT F..."
406,contract_0407,KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1-PROM...


In [8]:
df_documents = df_documents[
    ["document_id", "contract_id", "context", "context_group_id"]
].reset_index(drop=True)
df_documents

,document_id,contract_id,context,context_group_id
0,document_0001,contract_0001,EXHIBIT 10.6\n\n ...,context_41f7921a65
1,document_0002,contract_0002,Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...,context_d545018697
2,document_0003,contract_0003,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...,context_8cc65b1516
3,document_0004,contract_0004,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...,context_3b2346a2bc
4,document_0005,contract_0005,Exhibit 10.23 Corporate Address Fannin South P...,context_1d348d1f41
...,...,...,...,...
403,document_0404,contract_0404,Exhibit 99(h)(3) WHOLESALE MARKETING AGREEMENT...,context_089b029b55
404,document_0405,contract_0405,"Exhibit 1.1 400,000 Shares BLUEROCK RESIDE...",context_d621f3c0b1
405,document_0406,contract_0406,Exhibit 26\n\nJOINT FILING AGREEMENT\n\nPursua...,context_d46fd85099
406,document_0407,contract_0407,Exhibit 10.1\n\n\n\nPROMOTION AGREEMENT\n\nby ...,context_dc81522bbe


In [9]:
df_qas = pd.json_normalize(
    raw_data,
    record_path=["paragraphs", "qas"],
    meta=["title", "contract_id"]
)
df_qas

,answers,id,question,is_impossible,title,contract_id
0,"[{'text': 'DISTRIBUTOR AGREEMENT', 'answer_sta...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001
1,"[{'text': 'Distributor', 'answer_start': 244}]",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001
2,"[{'text': 'Electric City of Illinois L.L.C.', ...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001
3,"[{'text': 'Electric City of Illinois LLC', 'an...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001
4,"[{'text': 'Company', 'answer_start': 197}]",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001
...,...,...,...,...,...,...
22445,"[{'text': 'Company agrees, at its own expense,...",PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,False,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408
22446,[{'text': 'Such insurance policy shall be mai...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,False,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408
22447,[{'text': 'A copy of such insurance policy sha...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,False,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408
22448,[],PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,True,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408


In [10]:
df_qas["category_name"] = df_qas["question"].str.extract(f'"([^"]+)')
df_qas["category_id"] = (
    df_qas["category_name"]
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
df_qas["annotation_set_id"] = df_qas["contract_id"] + "__" + df_qas['category_id'].str.lower()

df_qas

,answers,id,question,is_impossible,title,contract_id,category_name,category_id,annotation_set_id
0,"[{'text': 'DISTRIBUTOR AGREEMENT', 'answer_sta...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001,Document Name,document_name,contract_0001__document_name
1,"[{'text': 'Distributor', 'answer_start': 244}]",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001,Parties,parties,contract_0001__parties
2,"[{'text': 'Electric City of Illinois L.L.C.', ...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001,Parties,parties,contract_0001__parties
3,"[{'text': 'Electric City of Illinois LLC', 'an...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001,Parties,parties,contract_0001__parties
4,"[{'text': 'Company', 'answer_start': 197}]",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Highlight the parts (if any) of this contract ...,False,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001,Parties,parties,contract_0001__parties
...,...,...,...,...,...,...,...,...,...
22445,"[{'text': 'Company agrees, at its own expense,...",PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,False,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408,Insurance,insurance,contract_0408__insurance
22446,[{'text': 'Such insurance policy shall be mai...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,False,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408,Insurance,insurance,contract_0408__insurance
22447,[{'text': 'A copy of such insurance policy sha...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,False,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408,Insurance,insurance,contract_0408__insurance
22448,[],PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Highlight the parts (if any) of this contract ...,True,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408,Covenant Not To Sue,covenant_not_to_sue,contract_0408__covenant_not_to_sue


In [11]:
df_categories = (
    df_qas[["category_id", "category_name", "question"]]
    .drop_duplicates(subset="category_id")
    .reset_index(drop=True)
)

df_categories

,category_id,category_name,question
0,document_name,Document Name,Highlight the parts (if any) of this contract ...
1,parties,Parties,Highlight the parts (if any) of this contract ...
2,agreement_date,Agreement Date,Highlight the parts (if any) of this contract ...
3,effective_date,Effective Date,Highlight the parts (if any) of this contract ...
4,expiration_date,Expiration Date,Highlight the parts (if any) of this contract ...
5,renewal_term,Renewal Term,Highlight the parts (if any) of this contract ...
6,notice_period_to_terminate_renewal,Notice Period To Terminate Renewal,Highlight the parts (if any) of this contract ...
7,governing_law,Governing Law,Highlight the parts (if any) of this contract ...
8,most_favored_nation,Most Favored Nation,Highlight the parts (if any) of this contract ...
9,non_compete,Non-Compete,Highlight the parts (if any) of this contract ...


In [12]:
df_annotation_sets = (
    df_qas.groupby(
        ["annotation_set_id", "contract_id", "category_id"]
    , as_index=False)["is_impossible"]
    .all()
)

df_annotation_sets

,annotation_set_id,contract_id,category_id,is_impossible
0,contract_0001__affiliate_license_licensee,contract_0001,affiliate_license_licensee,True
1,contract_0001__affiliate_license_licensor,contract_0001,affiliate_license_licensor,True
2,contract_0001__agreement_date,contract_0001,agreement_date,False
3,contract_0001__anti_assignment,contract_0001,anti_assignment,False
4,contract_0001__audit_rights,contract_0001,audit_rights,True
...,...,...,...,...
16723,contract_0408__third_party_beneficiary,contract_0408,third_party_beneficiary,True
16724,contract_0408__uncapped_liability,contract_0408,uncapped_liability,True
16725,contract_0408__unlimited_all_you_can_eat_license,contract_0408,unlimited_all_you_can_eat_license,True
16726,contract_0408__volume_restriction,contract_0408,volume_restriction,False


In [13]:
df_spans = df_qas[["answers", "id", "annotation_set_id"]].explode("answers")
df_spans = df_spans[df_spans["answers"].notna()].copy() # drops impossible rows
df_spans

,answers,id,annotation_set_id
0,"{'text': 'DISTRIBUTOR AGREEMENT', 'answer_star...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__document_name
1,"{'text': 'Distributor', 'answer_start': 244}",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties
2,"{'text': 'Electric City of Illinois L.L.C.', '...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties
3,"{'text': 'Electric City of Illinois LLC', 'ans...",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties
4,"{'text': 'Company', 'answer_start': 197}",LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties
...,...,...,...
22439,{'text': 'Said books and records shall be main...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__post_termination_services
22440,{'text': 'Company shall make said books availa...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__audit_rights
22445,"{'text': 'Company agrees, at its own expense, ...",PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__insurance
22446,{'text': 'Such insurance policy shall be main...,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__insurance


In [14]:
df_answer_fields = pd.json_normalize(df_spans["answers"])
df_answer_fields

,text,answer_start
0,DISTRIBUTOR AGREEMENT,44
1,Distributor,244
2,Electric City of Illinois L.L.C.,49574
3,Electric City of Illinois LLC,212
4,Company,197
...,...,...
22439,Said books and records shall be maintained for...,12067
22440,Company shall make said books available to Nor...,12196
22445,"Company agrees, at its own expense, to obtain ...",20750
22446,Such insurance policy shall be maintained wit...,21123


In [15]:
df_spans = pd.concat([df_spans.drop(columns="answers"), df_answer_fields], axis=1)
df_spans = df_spans.rename(columns={"id": "source_qa_id", "text": "answer_text"})
df_spans["answer_end"] = df_spans["answer_start"] + df_spans["answer_text"].str.len()
df_spans

,source_qa_id,annotation_set_id,answer_text,answer_start,answer_end
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__document_name,DISTRIBUTOR AGREEMENT,44,65
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties,Distributor,244,255
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties,Electric City of Illinois L.L.C.,49574,49606
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties,Electric City of Illinois LLC,212,241
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,contract_0001__parties,Company,197,204
...,...,...,...,...,...
22439,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__post_termination_services,Said books and records shall be maintained for...,12067,12195
22440,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__audit_rights,Company shall make said books available to Nor...,12196,12388
22445,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__insurance,"Company agrees, at its own expense, to obtain ...",20750,21071
22446,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,contract_0408__insurance,Such insurance policy shall be maintained wit...,21123,21228


In [16]:
ag_num_spans = df_spans.groupby("annotation_set_id").cumcount()
df_spans["span_id"] = (
        df_spans["annotation_set_id"] + "__span_" + ag_num_spans.astype(str).str.zfill(3)
)
df_spans = df_spans[[
    "span_id",
    "annotation_set_id",
    "source_qa_id",
    "answer_text",
    "answer_start",
    "answer_end",
]].reset_index(drop=True)
df_spans

,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end
0,contract_0001__document_name__span_000,contract_0001__document_name,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,DISTRIBUTOR AGREEMENT,44,65
1,contract_0001__parties__span_000,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Distributor,244,255
2,contract_0001__parties__span_001,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Electric City of Illinois L.L.C.,49574,49606
3,contract_0001__parties__span_002,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Electric City of Illinois LLC,212,241
4,contract_0001__parties__span_003,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Company,197,204
...,...,...,...,...,...,...
11175,contract_0408__post_termination_services__span...,contract_0408__post_termination_services,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Said books and records shall be maintained for...,12067,12195
11176,contract_0408__audit_rights__span_000,contract_0408__audit_rights,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Company shall make said books available to Nor...,12196,12388
11177,contract_0408__insurance__span_000,contract_0408__insurance,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,"Company agrees, at its own expense, to obtain ...",20750,21071
11178,contract_0408__insurance__span_001,contract_0408__insurance,PerformanceSportsBrandsInc_20110909_S-1_EX-10....,Such insurance policy shall be maintained wit...,21123,21228


# Verifying Counts

In [18]:
normalized_data = normalize_cuad(raw_data)

In [19]:
totalContracts = 408
totalDocuments = 408
totalCategories = 41
totalAnnotationSets = 16728
totalSpans = 11180

contracts = normalized_data["contracts"]
documents = normalized_data["documents"]
categories = normalized_data["categories"]
annotation_sets = normalized_data["annotation_sets"]
spans = normalized_data["spans"]

Confirm these exact totals:
- [x] 408 Contracts
- [x] 408 Documents
- [x] 41 Categories
- [x] 16,728 Annotation Sets
- [x] 11,180 Spans

In [20]:
print(f"{totalContracts} Contracts \t\t\t\t{"PASSED" if contracts.shape[0] == totalContracts else "FAILED"}")
print(f"{totalDocuments} Documents \t\t\t\t{"PASSED" if documents.shape[0] == totalDocuments else "FAILED"}")
print(f"{totalCategories} Categories \t\t\t\t{"PASSED" if categories.shape[0] == totalCategories else "FAILED"}")
print(f"{totalAnnotationSets} AnnotationSets \t\t{"PASSED" if annotation_sets.shape[0] == totalAnnotationSets else "FAILED"}")
print(f"{totalSpans} Spans \t\t\t\t{"PASSED" if spans.shape[0] == totalSpans else "FAILED"}")


408 Contracts 				PASSED
408 Documents 				PASSED
41 Categories 				PASSED
16728 AnnotationSets 		PASSED
11180 Spans 				PASSED
